In [2]:
import pandas as pd
df = pd.read_stata("../data/raw/qlfs-2026-q1-v1.dta", convert_categoricals=False)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (62819, 201)


,UQNO,SURVEYDATE,PERSONNO,Q12NIGHTS,Q13GENDER,Q14DAY,Q14MONTH,Q14YEAR,Q14AGE,Q15POPULATION,...,Hrswrk,Education_status,Neet,Graduates,Ste_icse93,Ste_icse18_authority,Ste_icse18,Ste_icse18_risk,Weight,uqno
0,160100880000001801,1022026,1,1,1,2,7,1996,29,4,...,50.0,6,2.0,3,1.0,4.0,8.0,5.0,2209.194153,1
1,160100880000001801,1022026,2,1,2,16,2,1998,28,4,...,40.0,6,2.0,4,1.0,4.0,8.0,5.0,2209.194153,1
2,160100880000001801,1022026,3,1,1,24,1,2024,2,4,...,NaN,1,NaN,1,NaN,NaN,NaN,NaN,2209.194153,1
3,160100880000005801,1022026,2,1,2,10,7,1977,48,4,...,45.0,5,2.0,2,1.0,4.0,8.0,5.0,1231.493695,2
4,160100880000005801,1022026,3,1,2,23,5,2016,9,4,...,NaN,2,NaN,1,NaN,NaN,NaN,NaN,1231.493695,2


In [3]:
# List ALL column names so we can find Province & Employment columns
for col in df.columns:
    print(col)


UQNO
SURVEYDATE
PERSONNO
Q12NIGHTS
Q13GENDER
Q14DAY
Q14MONTH
Q14YEAR
Q14AGE
Q15POPULATION
Q16MARITALSTATUS
Q17EDUCATION
Q18FIELD
Q19ATTE
Q110EDUI
Q111APPRENTICE
Q112TRAIN
Q20SELFRESPOND
Q24APDWRK
Q24BOWNBUSNS
Q24CUNPDWRK
Q25APDWRK
Q27RSNABSENT
Q27OFFSEASON
Q27ATIME
Q27BRECPAY
Q2801CROPS
Q2802ANIMALS
Q2803FISHING
Q2804NONE
Q2901CROPS
Q2902ANIMALS
Q2903FISHING
Q2904JOB
Q210MARKET
Q211HIRE
Q212SUBINDUSTRY
Q31ALOOKWRK
Q31BSTARTBUSNS
Q3201REGISTER
Q3202ENQUIRE
Q3203JOBADS
Q3204JOBSEARCH
Q3205ASSISTANCE
Q3206STARTBUSNS
Q3207CASUAL
Q3208FINASSIST
Q33HAVEJOB
Q33AWHNSTART
Q34WANTTOWRK
Q35YNOTWRK
Q36TIMESEEK
Q37ACTPRIORJOBSEEK
Q38RSNNOTSEEK
Q39JOBOFFER
Q311RSNNOTAVAILABLE
Q311bWHNSTART
Q312EVERWRK
Q313TIMEUNEMPLOY
Q314RSNSTOPWRK
Q315PREVOCCUPATION
Q316PREVINDUSTRY
Q317WRK4WHOM
Q319aODDJOBS
Q319bINHHPERS
Q319cNOTHHPERS
Q319dCHARITY
Q319eUIF
Q319fSAVINGS
Q319gPENSION
Q319hGRANTS
Q319iWELFARE
Q319jOTHER
Q41MULTIPLEJOBS
Q42OCCUPATION
Q43INDUSTRY
Q44YEARSTART
Q44MONTHSTART
Q45WRK4WHOM
Q45aDECISON
Q45

In [4]:
# Search for columns related to province, employment status, and sector
keywords = ['prov', 'stat', 'sector', 'indus', 'employ', 'unemp', 'work']

for col in df.columns:
    lower_col = col.lower()
    if any(kw in lower_col for kw in keywords):
        print(col)

Q16MARITALSTATUS
Q212SUBINDUSTRY
Q313TIMEUNEMPLOY
Q316PREVINDUSTRY
Q43INDUSTRY
Q416NRWORKERS
Province
Status
Lfs_Status
Unempl_Status
Indus
PrevIndus
Sector
Long_term_unempl
Education_status


In [ ]:

print("=== Province ===")
print(df['Province'].value_counts().sort_index())

print("\n=== Status ===")
print(df['Status'].value_counts().sort_index())

print("\n=== Lfs_Status ===")
print(df['Lfs_Status'].value_counts().sort_index())

print("\n=== Sector ===")
print(df['Sector'].value_counts().sort_index())

print("\n=== Education_status ===")
print(df['Education_status'].value_counts().sort_index())

=== Province ===
Province
1     6815
2     8089
3     2532
4     4045
5    11188
6     3603
7    13495
8     5383
9     7669
Name: count, dtype: int64

=== Status ===
Status
1.0    15567
2.0     7518
3.0     3642
4.0    19146
Name: count, dtype: int64

=== Lfs_Status ===
Lfs_Status
1.0    15567
2.0     7518
3.0     3642
4.0      958
5.0       44
6.0    18144
Name: count, dtype: int64

=== Sector ===
Sector
1.0    11061
2.0     3384
3.0     1122
Name: count, dtype: int64

=== Education_status ===
Education_status
1     8830
2    10762
3     3157
4    19719
5    14319
6     5422
7      610
Name: count, dtype: int64


In [ ]:

print("=" * 60)
print("OFFICIAL UNEMPLOYMENT RATE BY PROVINCE — Q1 2026")
print("=" * 60)

province_names = {
    1: 'Western Cape',
    2: 'Northern Cape',
    3: 'Free State',
    4: 'Eastern Cape',
    5: 'KwaZulu-Natal',
    6: 'North West',
    7: 'Gauteng',
    8: 'Mpumalanga',
    9: 'Limpopo'
}

results = []

for prov_code, prov_name in province_names.items():
    prov_df = df[df['Province'] == prov_code]
    
    # Count Employed (Lfs_Status == 1) and Unemployed (Lfs_Status == 2)
    employed = (prov_df['Lfs_Status'] == 1).sum()
    unemployed = (prov_df['Lfs_Status'] == 2).sum()
    labour_force = employed + unemployed
    
    if labour_force > 0:
        unemp_rate = (unemployed / labour_force) * 100
    else:
        unemp_rate = 0
    
    results.append({
        'Province': prov_name,
        'Employed': employed,
        'Unemployed': unemployed,
        'Labour_Force': labour_force,
        'Unemployment_Rate_%': round(unemp_rate, 1)
    })


results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Unemployment_Rate_%', ascending=False)
print(results_df.to_string(index=False))


print("\n" + "=" * 60)
kzn_row = results_df[results_df['Province'] == 'KwaZulu-Natal'].iloc[0]
print(f"KZN UNEMPLOYMENT RATE: {kzn_row['Unemployment_Rate_%']}%")
print(f"KZN Unemployed: {kzn_row['Unemployed']:,}")
print(f"KZN Labour Force: {kzn_row['Labour_Force']:,}")
print("=" * 60)

OFFICIAL UNEMPLOYMENT RATE BY PROVINCE — Q1 2026
     Province  Employed  Unemployed  Labour_Force  Unemployment_Rate_%
Northern Cape      1423        1027          2450                 41.9
 Eastern Cape       954         640          1594                 40.2
   North West       697         375          1072                 35.0
      Gauteng      4017        2152          6169                 34.9
   Mpumalanga      1298         696          1994                 34.9
   Free State       533         247           780                 31.7
      Limpopo      1579         712          2291                 31.1
KwaZulu-Natal      2500        1027          3527                 29.1
 Western Cape      2566         642          3208                 20.0

KZN UNEMPLOYMENT RATE: 29.1%
KZN Unemployed: 1,027
KZN Labour Force: 3,527


In [ ]:

province_names = {
    1: 'Western Cape', 2: 'Northern Cape', 3: 'Free State',
    4: 'Eastern Cape', 5: 'KwaZulu-Natal', 6: 'North West',
    7: 'Gauteng', 8: 'Mpumalanga', 9: 'Limpopo'
}

results = []

for prov_code, prov_name in province_names.items():
    prov_df = df[df['Province'] == prov_code]
    
    # Weighted counts — multiply each person by their survey weight
    employed = prov_df.loc[prov_df['Lfs_Status'] == 1, 'Weight'].sum()
    unemployed = prov_df.loc[prov_df['Lfs_Status'] == 2, 'Weight'].sum()
    labour_force = employed + unemployed
    
    unemp_rate = (unemployed / labour_force * 100) if labour_force > 0 else 0
    
    results.append({
        'Province': prov_name,
        'Employed_M': round(employed / 1_000_000, 2),
        'Unemployed_M': round(unemployed / 1_000_000, 2),
        'Unemployment_Rate_%': round(unemp_rate, 1)
    })

results_df = pd.DataFrame(results).sort_values('Unemployment_Rate_%', ascending=False)
print(results_df.to_string(index=False))

# National average
total_employed = df.loc[df['Lfs_Status'] == 1, 'Weight'].sum()
total_unemployed = df.loc[df['Lfs_Status'] == 2, 'Weight'].sum()
national_rate = total_unemployed / (total_employed + total_unemployed) * 100
print(f"\nNATIONAL UNEMPLOYMENT RATE: {national_rate:.1f}%")

     Province  Employed_M  Unemployed_M  Unemployment_Rate_%
Northern Cape        1.35          1.07                 44.3
 Eastern Cape        0.77          0.46                 37.3
   Mpumalanga        1.25          0.70                 35.9
   North West        0.90          0.49                 35.2
      Gauteng        5.18          2.65                 33.9
      Limpopo        1.53          0.70                 31.3
KwaZulu-Natal        2.74          1.23                 31.1
   Free State        0.31          0.13                 30.2
 Western Cape        2.93          0.71                 19.5

NATIONAL UNEMPLOYMENT RATE: 32.5%


In [ ]:

kzn = df[df['Province'] == 5].copy()

print("=" * 70)
print(f"KZN DEEP DIVE — Total respondents: {len(kzn):,}")
print("=" * 70)


print("\n--- KZN Employment by Sector ---")
kzn_employed = kzn[kzn['Lfs_Status'] == 1]
sector_breakdown = kzn_employed.groupby('Sector')['Weight'].sum().sort_values(ascending=False)
print(sector_breakdown)


print("\n--- KZN Employment by Industry (Indus) ---")
industry_breakdown = kzn_employed.groupby('Indus')['Weight'].sum().sort_values(ascending=False).head(10)
print(industry_breakdown)


print("\n--- KZN Employment by Education Status ---")
edu_breakdown = kzn_employed.groupby('Education_status')['Weight'].sum().sort_values(ascending=False)
print(edu_breakdown)


kzn['Age_Group'] = pd.cut(
    kzn['Q14AGE'],
    bins=[0, 14, 24, 34, 44, 54, 64, 120],
    labels=['0-14', '15-24 (Youth)', '25-34 (Youth)', '35-44', '45-54', '55-64', '65+']
)


print("\n--- KZN Unemployment Rate by Age Group ---")
age_stats = []
for group in kzn['Age_Group'].cat.categories:
    group_df = kzn[kzn['Age_Group'] == group]
    employed = group_df.loc[group_df['Lfs_Status'] == 1, 'Weight'].sum()
    unemployed = group_df.loc[group_df['Lfs_Status'] == 2, 'Weight'].sum()
    lf = employed + unemployed
    rate = (unemployed / lf * 100) if lf > 0 else 0
    age_stats.append({'Age_Group': group, 'Unemployment_Rate_%': round(rate, 1), 'Labour_Force': int(lf)})

print(pd.DataFrame(age_stats).to_string(index=False))

KZN DEEP DIVE — Total respondents: 11,188

--- KZN Employment by Sector ---
Sector
1.0    1.906221e+06
2.0    6.426370e+05
3.0    1.915506e+05
Name: Weight, dtype: float64

--- KZN Employment by Industry (Indus) ---
Indus
9.0     625660.903697
6.0     552506.221752
8.0     447397.535685
3.0     354359.833361
7.0     216376.572527
5.0     211971.178493
10.0    191550.581326
1.0     125115.651053
4.0       8250.294536
2.0       7220.222552
Name: Weight, dtype: float64

--- KZN Employment by Education Status ---
Education_status
5    1.107590e+06
4    8.432933e+05
6    5.282039e+05
2    1.412711e+05
3    6.989832e+04
1    2.934578e+04
7    2.080644e+04
Name: Weight, dtype: float64

--- KZN Unemployment Rate by Age Group ---
    Age_Group  Unemployment_Rate_%  Labour_Force
         0-14                  0.0             0
15-24 (Youth)                 64.6        402304
25-34 (Youth)                 40.6       1397237
        35-44                 23.4       1149867
        45-54           

In [9]:
kzn = df[df['Province'] == 5].copy()

# Age groups
kzn['Age_Group'] = pd.cut(
    kzn['Q14AGE'],
    bins=[0, 14, 24, 34, 44, 54, 64, 120],
    labels=['0-14', '15-24 Youth', '25-34 Youth', '35-44', '45-54', '55-64', '65+']
)

print("=" * 55)
print("KZN UNEMPLOYMENT BY AGE GROUP")
print("=" * 55)
for group in kzn['Age_Group'].cat.categories:
    g = kzn[kzn['Age_Group'] == group]
    emp = g.loc[g['Lfs_Status'] == 1, 'Weight'].sum()
    unemp = g.loc[g['Lfs_Status'] == 2, 'Weight'].sum()
    lf = emp + unemp
    rate = (unemp / lf * 100) if lf > 0 else 0
    print(f"{group:15s}  {rate:5.1f}%   LF: {int(lf):>8,}")

print("\n" + "=" * 55)
print("KZN EMPLOYMENT BY EDUCATION STATUS")
print("=" * 55)
kzn_emp = kzn[kzn['Lfs_Status'] == 1]
edu = kzn_emp.groupby('Education_status')['Weight'].sum().sort_values(ascending=False)
print(edu.to_string())

KZN UNEMPLOYMENT BY AGE GROUP
0-14               0.0%   LF:        0
15-24 Youth       64.6%   LF:  402,304
25-34 Youth       40.6%   LF: 1,397,237
35-44             23.4%   LF: 1,149,867
45-54             16.3%   LF:  744,304
55-64              6.2%   LF:  262,143
65+                0.0%   LF:   18,900

KZN EMPLOYMENT BY EDUCATION STATUS
Education_status
5    1.107590e+06
4    8.432933e+05
6    5.282039e+05
2    1.412711e+05
3    6.989832e+04
1    2.934578e+04
7    2.080644e+04


In [ ]:

print("Education code meanings from the data dictionary:")
print(df['Education_status'].value_counts().sort_index())


print("\nSector code meanings:")
print(df['Sector'].value_counts().sort_index())

Education code meanings from the data dictionary:
Education_status
1     8830
2    10762
3     3157
4    19719
5    14319
6     5422
7      610
Name: count, dtype: int64

Sector code meanings:
Sector
1.0    11061
2.0     3384
3.0     1122
Name: count, dtype: int64


In [ ]:
import pandas as pd


with pd.io.stata.StataReader("../data/raw/qlfs-2026-q1-v1.dta") as reader:
    value_labels = reader.value_labels()
    label_sets = reader.variable_labels()


for labelset_name in ['Education_status', 'Sector', 'Indus', 'Status', 'Lfs_Status']:
    if labelset_name in value_labels:
        print(f"\n=== {labelset_name} ===")
        for code, label in value_labels[labelset_name].items():
            print(f"  {code}  →  {label}")
            


=== Education_status ===
  1  →  No schooling
  2  →  Less than primary completed
  3  →  Primary completed
  4  →  Secondary not completed
  5  →  Secondary completed
  6  →  Tertiary
  7  →  Other

=== Sector ===
  1  →  Formal sector
  2  →  Informal
  3  →  Households

=== Indus ===
  1  →  Agriculture; hunting; forestry and fishing
  2  →  Mining and quarrying
  3  →  Manufacturing
  4  →  Electricity; gas and water supply
  5  →  Construction
  6  →  Wholesale and retail trade
  7  →  Transport; storage and communication
  8  →  Financial intermediation; insurance; real estate and business services
  9  →  Community; social and personal services
  10  →  Private households
  11  →  Other

=== Status ===
  1  →  Employed
  2  →  Unemployed
  3  →  Discouraged job seeker
  4  →  Other not economically active

=== Lfs_Status ===
  1  →  Employed
  2  →  Unemployed
  3  →  Discouraged jobseekers
  4  →  Available jobseekers
  5  →  Unavailable jobseekers
  6  →  Other outside the la

C:\Users\shezi\AppData\Local\Temp\ipykernel_27460\1410041157.py:6: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  value_labels = reader.value_labels()


In [ ]:
kzn = df[df['Province'] == 5].copy()


sector_map = {1: 'Formal sector', 2: 'Informal sector', 3: 'Households'}
industry_map = {
    1: 'Agriculture, forestry & fishing',
    2: 'Mining & quarrying',
    3: 'Manufacturing',
    4: 'Electricity, gas & water',
    5: 'Construction',
    6: 'Wholesale & retail trade',
    7: 'Transport & communication',
    8: 'Financial & business services',
    9: 'Community & social services',
    10: 'Private households'
}


kzn['Age_Group'] = pd.cut(kzn['Q14AGE'],
    bins=[0, 14, 24, 34, 44, 54, 64, 120],
    labels=['0-14', '15-24', '25-34', '35-44', '45-54', '55-64', '65+'])


print("=" * 65)
print("KWAZULU-NATAL UNEMPLOYMENT ANALYSIS — Q1 2026")
print("=" * 65)


kzn_emp = kzn.loc[kzn['Lfs_Status'] == 1, 'Weight'].sum()
kzn_unemp = kzn.loc[kzn['Lfs_Status'] == 2, 'Weight'].sum()
kzn_lf = kzn_emp + kzn_unemp
print(f"\n1. OVERALL")
print(f"   KZN Unemployment Rate:   {(kzn_unemp/kzn_lf*100):.1f}%")
print(f"   Employed (weighted):     {kzn_emp:,.0f}")
print(f"   Unemployed (weighted):   {kzn_unemp:,.0f}")


print(f"\n2. WHERE KZN WORKS (top sectors)")
kzn_employed = kzn[kzn['Lfs_Status'] == 1]
sector_w = kzn_employed.groupby('Sector')['Weight'].sum().sort_values(ascending=False)
for code, w in sector_w.items():
    print(f"   {sector_map.get(int(code), code):20s}  {w:,.0f}")


print(f"\n3. TOP 5 INDUSTRIES IN KZN")
industry_w = kzn_employed.groupby('Indus')['Weight'].sum().sort_values(ascending=False).head(5)
for code, w in industry_w.items():
    print(f"   {industry_map.get(int(code), code):35s}  {w:,.0f}")


print(f"\n4. YOUTH UNEMPLOYMENT IN KZN")
for group in ['15-24', '25-34']:
    g = kzn[kzn['Age_Group'] == group]
    emp = g.loc[g['Lfs_Status'] == 1, 'Weight'].sum()
    unemp = g.loc[g['Lfs_Status'] == 2, 'Weight'].sum()
    lf = emp + unemp
    print(f"   {group}:  {(unemp/lf*100):.1f}%  (LF: {int(lf):,})")

print("\n" + "=" * 65)

KWAZULU-NATAL UNEMPLOYMENT ANALYSIS — Q1 2026

1. OVERALL
   KZN Unemployment Rate:   31.1%
   Employed (weighted):     2,740,409
   Unemployed (weighted):   1,234,348

2. WHERE KZN WORKS (top sectors)
   Formal sector         1,906,221
   Informal sector       642,637
   Households            191,551

3. TOP 5 INDUSTRIES IN KZN
   Community & social services          625,661
   Wholesale & retail trade             552,506
   Financial & business services        447,398
   Manufacturing                        354,360
   Transport & communication            216,377

4. YOUTH UNEMPLOYMENT IN KZN
   15-24:  64.6%  (LF: 402,304)
   25-34:  40.6%  (LF: 1,397,237)

